In [73]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import statistics
import math

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import RocCurveDisplay, roc_auc_score, log_loss, brier_score_loss, precision_recall_curve, average_precision_score, precision_score, recall_score, confusion_matrix

In [83]:
RETRIEVAL_DIR = Path("../output/retrieval")
DATA_DIR = Path("../data/gold_data")

retrieval_files = list((RETRIEVAL_DIR/"bm25_rerank").rglob("*.json"))
gold_full_df = pd.read_csv(DATA_DIR / "gold_standard_processed.csv")
gold_no_graphics_df =  pd.read_csv(DATA_DIR / "gold_standard_processed_no_graphics.csv")

# Create Dataset

In [116]:
gold_pages_by_report = gold_full_df.groupby("report_id")["page"].agg(lambda x: set(x.dropna())).to_dict()
results = []

for retrieval_file in retrieval_files:
    # if retrieval_file.stem.split("_")[0] != "apollo medical holdings":
    #     continue
    
    with open(retrieval_file, "r", encoding="utf-8") as f:
        rdata = json.load(f)
    
    
    retrieval_method = rdata.get("method")
    
    report_id = rdata.get("report_id")
    
    retrieved_results = rdata.get("items", [])
    retrieved_pages = set()
    retrieved_scores = []
    
    for item in retrieved_results:
        metadata = item.get("metadata", {})
        page_numbers = metadata.get("page_no")
        score = item.get("score")
        retrieved_pages.add(page_numbers)
        retrieved_scores.append(score)
        retrieved_scores.sort()
        retrieved_logits = [math.log(s / (1 - s)) for s in retrieved_scores]
        retrieved_logits.sort()
    
    gold_pages = gold_pages_by_report[report_id]
    
    if gold_pages and gold_pages.issubset(retrieved_pages):
        found_all_gold_pages = True
        gold_exists = True
    elif gold_pages:
        found_all_gold_pages = False
        gold_exists = True
    else:
        found_all_gold_pages = False
        gold_exists = False
    
    # keep = at least one gold page is among the retrieved top-k pages (something correct to preserve)
    keep = len(gold_pages & retrieved_pages) > 0
    
    results.append({
        "report_id": report_id,
        "1st": retrieved_scores[-1] if len(retrieved_scores) > 0 else None,
        "2nd": retrieved_scores[-2] if len(retrieved_scores) > 1 else None,
        "3rd": retrieved_scores[-3] if len(retrieved_scores) > 2 else None,
        "4th": retrieved_scores[-4] if len(retrieved_scores) > 3 else None,
        "5th": retrieved_scores[-5] if len(retrieved_scores) > 4 else None,
        "mean_score": statistics.mean(retrieved_scores) if retrieved_scores else None,
        "score_gap": retrieved_scores[-1] - retrieved_scores[-2] if len(retrieved_scores) > 1 else None,
        # "1st": retrieved_logits[-1] if len(retrieved_logits) > 0 else None,
        # "2nd": retrieved_logits[-2] if len(retrieved_logits) > 1 else None,
        # "3rd": retrieved_logits[-3] if len(retrieved_logits) > 2 else None,
        # "4th": retrieved_logits[-4] if len(retrieved_logits) > 3 else None,
        # "5th": retrieved_logits[-5] if len(retrieved_logits) > 4 else None,
        # "mean_score": statistics.mean(retrieved_logits) if retrieved_logits else None,
        # "score_gap": retrieved_logits[-1] - retrieved_logits[-2] if len(retrieved_logits) > 1 else None,
        "found_all_gold_pages": found_all_gold_pages,
        "gold_exists": gold_exists,
        "keep": keep,
    })
    
df = pd.DataFrame(results)

print(f"Number of reports with KPIs: {len(df.query('gold_exists == True'))}")
print(f"Number of reports with KPIs and correct pages: {len(df.query('gold_exists == True and found_all_gold_pages == True'))}")
print(f"Number of reports with KPIs but incorrect pages: {len(df.query('gold_exists == True and found_all_gold_pages == False'))}")
print(f"Number of reports without KPIs: {len(df.query('gold_exists == False'))}")
print(f"Number of reports to keep (>= 1 gold page retrieved): {len(df.query('keep == True'))}")

df

Number of reports with KPIs: 69
Number of reports with KPIs and correct pages: 66
Number of reports with KPIs but incorrect pages: 3
Number of reports without KPIs: 70
Number of reports to keep (>= 1 gold page retrieved): 68


,report_id,1st,2nd,3rd,4th,5th,mean_score,score_gap,found_all_gold_pages,gold_exists,keep
0,acuity brands inc_2022,0.982014,0.904651,0.808067,0.577495,0.531209,0.760687,0.077363,True,True,True
1,addtech_2022,0.468791,0.307358,0.191933,0.132964,0.080357,0.236281,0.161433,True,True,True
2,aixtron_2020,0.966914,0.766294,0.222700,0.222700,0.125923,0.460906,0.200620,False,True,True
3,allfunds group_2021,0.798187,0.268941,0.173288,0.071591,0.031144,0.268630,0.529245,True,True,True
4,Allianz_2022,0.955319,0.718594,0.268941,0.233706,0.191933,0.473699,0.236725,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
134,varta ag_2021,0.766294,0.090093,0.042088,0.016915,0.016915,0.186461,0.676201,True,True,True
135,webuild_2019,0.743168,0.320821,0.085099,0.035145,0.007121,0.238271,0.422347,True,True,True
136,western union_2019,0.156105,0.060087,0.047426,0.017986,0.014957,0.059312,0.096018,False,False,False
137,xvivo perfusion_2020,0.005555,0.003377,0.003377,0.002473,0.002473,0.003451,0.002178,False,False,False


In [135]:
df.query('gold_exists == True and keep == False')

,report_id,1st,2nd,3rd,4th,5th,mean_score,score_gap,found_all_gold_pages,gold_exists,keep
66,huhtamaki_2018,0.050331,0.022977,0.014957,0.010328,0.009126,0.021544,0.027353,False,True,False


In [132]:
n = df["keep"].sum()                      # 68 keep reports

expected_recall = n / (n + 1)             # 1) rank argument: a new gold report is the lowest of n+1 with prob. 1/(n+1)
recall_95 = 0.05 ** (1 / n)               # 2) with 95% confidence, true recall is at least this
recall_99 = 0.01 ** (1 / n)               # 2) with 99% confidence, true recall is at least this

print(f"Expected recall on new reports: >= {expected_recall:.4f}")
print(f"Recall with 95% confidence:     >= {recall_95:.4f}")
print(f"Recall with 99% confidence:     >= {recall_99:.4f}")

Expected recall on new reports: >= 0.9855
Recall with 95% confidence:     >= 0.9569
Recall with 99% confidence:     >= 0.9345


In [134]:
df.loc[df["keep"] == True, "1st"].sort_values()

27     0.027585
55     0.033086
93     0.148047
53     0.156105
102    0.173288
         ...   
32     0.980876
0      0.982014
56     0.988313
124    0.990292
22     0.993710
Name: 1st, Length: 68, dtype: float64

In [125]:
# Minimum top score among the reports to "keep" == True, meaning reports where at least one gold page were found by the retriever (with fixed settings -- no hardware noise etc.)
import math

def round_down(value, decimals=2):
    factor = 10 ** decimals
    return math.floor(value * factor) / factor

t_0 = df.loc[df["keep"] == True, "1st"].min()

threshold = round_down(t_0)